# The Bernstein-Vazirani algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/bernstein-vazirani.ipynb)

Read a hidden bitstring in a single query, where a classical computer needs one query per bit.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [The Bernstein-Vazirani algorithm](https://zksf.org/blog/bernstein-vazirani-algorithm/)


## The idea

A hidden string `s` lives inside an oracle. Ask it about an input `x` and it returns the bitwise dot product of `s` and `x`.

Classically you recover `s` one bit at a time: ask about `1000`, then `0100`, and so on, one query per bit. Bernstein-Vazirani gets the whole string in **one** query, by putting every input into superposition and letting interference do the extraction.

The hidden string here is `1011`.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

s = '1011'                   # the hidden string, unknown to the algorithm
n = len(s)

qc = QuantumCircuit(n + 1, n)
qc.x(n)                      # ancilla into |->
qc.h(n)
qc.h(range(n))               # every input at once

for i, bit in enumerate(reversed(s)):   # the oracle
    if bit == '1':
        qc.cx(i, n)

qc.h(range(n))               # interfere
qc.measure(range(n), range(n))
print(qc.draw(output='text'))


## What happened when this ran

Clifford throughout, so this ran exactly on the stabilizer engine.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "5a495a94476e45b0"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

Every shot returned `1011`, the hidden string, read from a single query. A classical computer would have needed four separate queries to learn those four bits.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify 5a495a94476e45b0
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000)
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/bernstein-vazirani-algorithm/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
